# 06 – Rule Agent (DQ & Business Rules Registry)

The **RuleAgent** manages a local in-memory rule registry:  
- **List** existing DQ and business rules  
- **Create** new rules from natural language  
- **Evaluate** rules against data products  

Seeded with 4 starter rules. No external dependencies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.rule_agent import RuleAgent, RULE_REGISTRY
from core.base_agent import AgentRequest

agent = RuleAgent()

## 1. List all seed rules

In [ ]:
result = agent.execute(AgentRequest(query='list all rules'))

print(result.message)
print()
for rule in result.data:
    print(f"  [{rule['id']}] [{rule['type']}] {rule['name']}")
    print(f"    asset     : {rule['asset']}")
    print(f"    expression: {rule['expression']}")
    print(f"    severity  : {rule['severity']}")
    print(f"    products  : {rule['products']}")
    print()

## 2. Filter rules by product

In [ ]:
result = agent.execute(AgentRequest(query='list rules', data_products=['retention']))
print(f'Rules for retention: {len(result.data)}')
for r in result.data:
    print(f"  {r['id']}: {r['name']}")

## 3. Create a data quality rule

In [ ]:
req_create = AgentRequest(
    query='Create a new data quality rule for bookings completeness',
    data_products=['bookings'],
    context={
        'rule_name': 'Bookings Null Check',
        'asset': 'analytics.bookings_fact',
        'dimension': 'completeness',
        'expression': 'null_count / total < 0.005',
        'severity': 'High',
    },
)
result = agent.execute(req_create)

print('Message:', result.message)
print('New rule:')
for k, v in result.data.items():
    print(f'  {k}: {v}')

## 4. Create a business rule

In [ ]:
req_br = AgentRequest(
    query='Define a business threshold rule: CAC payback must not exceed 24 months',
    data_products=['cac'],
    context={
        'rule_name': 'CAC Payback SLA',
        'asset': 'analytics.cac_metrics',
        'expression': 'payback_months <= 24',
        'threshold': 24,
        'severity': 'High',
    },
)
result = agent.execute(req_br)
print(result.message)
print(f"Rule type: {result.data['type']}")
print(f"Rule ID  : {result.data['id']}")

## 5. Evaluate rules

In [ ]:
result = agent.execute(AgentRequest(query='evaluate all rules'))

print(result.message)
print(f"  Passed : {result.metadata['passed']}")
print(f"  Failed : {result.metadata['failed']}")
print(f"  Skipped: {result.metadata['skipped']}")

print('\nEvaluation details:')
for r in result.data:
    icon = '' if r['passed'] else ''
    print(f'  {icon} {r["rule_id"]}: {r["rule_name"]} ({r["status"]})')

## 6. Inspect the live rule registry

In [ ]:
print(f'Total rules in registry: {len(RULE_REGISTRY)}')
print()
dq_rules = [r for r in RULE_REGISTRY.values() if r['type'] == 'data_quality']
br_rules = [r for r in RULE_REGISTRY.values() if r['type'] == 'business_rule']
print(f'  Data Quality rules : {len(dq_rules)}')
print(f'  Business rules     : {len(br_rules)}')